# Qwen 3 30B via curl

This notebook targets the OpenAI-compatible vLLM endpoint at `http://spark-0240:8000/v1`.

Run the Python cell below to:
- verify the available model list
- build a shell-ready `curl` command for chat completions
- call the same endpoint with `curl` from Python using `subprocess`

Direct shell example:

```bash
curl -s http://spark-0240:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d @- <<'JSON'
{
  "model": "Qwen/Qwen3-Coder-30B-A3B-Instruct-FP8",
  "messages": [
    {"role": "system", "content": "You are a precise coding assistant."},
    {"role": "user", "content": "Write a Python function that reverses a string."}
  ],
  "temperature": 0.2,
  "max_tokens": 256,
  "stream": false
}
JSON
```

In [ ]:
import json
import shlex
import subprocess

BASE_URL = "http://spark-0240:8000/v1"
MODEL = "Qwen/Qwen3.6-35B-A3B"


def curl_models(base_url: str = BASE_URL) -> dict:
    result = subprocess.run(
        ["curl", "-s", f"{base_url}/models"],
        capture_output=True,
        text=True,
        check=True,
    )
    data = json.loads(result.stdout)
    print(json.dumps(data, indent=2))
    return data


def build_curl_chat_command(
    message: str,
    *,
    system: str = "You are a precise coding assistant.",
    base_url: str = BASE_URL,
    model: str = MODEL,
    temperature: float = 0.2,
    max_tokens: int = 512,
    stream: bool = False,
) -> str:
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": message},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": stream,
    }
    command = [
        "curl",
        "-s",
        f"{base_url}/chat/completions",
        "-H",
        "Content-Type: application/json",
        "-d",
        json.dumps(payload),
    ]
    return shlex.join(command)


def extract_assistant_text(response: dict) -> str:
    try:
        message = response["choices"][0]["message"]
    except (KeyError, IndexError, TypeError):
        return ""

    return (
        message.get("content")
        or message.get("reasoning")
        or message.get("refusal")
        or ""
    )


def curl_chat(
    message: str,
    *,
    system: str = "You are a precise coding assistant.",
    base_url: str = BASE_URL,
    model: str = MODEL,
    temperature: float = 0.2,
    max_tokens: int = 512,
    extra_messages: list[dict] | None = None,
    extra_args: list[str] | None = None,
) -> dict:
    messages = [{"role": "system", "content": system}]
    if extra_messages:
        messages.extend(extra_messages)
    messages.append({"role": "user", "content": message})

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": False,
    }

    command = [
        "curl",
        "-s",
        f"{base_url}/chat/completions",
        "-H",
        "Content-Type: application/json",
        "-d",
        json.dumps(payload),
    ]

    if extra_args:
        command[1:1] = extra_args

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=True,
    )
    data = json.loads(result.stdout)
    print(json.dumps(data, indent=2))

    assistant_text = extract_assistant_text(data)
    if assistant_text:
        print("\nExtracted assistant text:\n")
        print(assistant_text)

    return data


models_response = curl_models()

sample_command = build_curl_chat_command(
    "Reply with exactly the word ready.",
    max_tokens=16,
    temperature=0,
    stream=False,
)

print("\nShell-ready curl command:\n")
print(sample_command)

# Run this when you want an actual chat completion from the notebook:
# response = curl_chat("Explain how to add a Home Assistant sensor entity.")

In [ ]:
response = curl_chat(
    "Reply with exactly the word ready.",
    max_tokens=16,
    temperature=0,
    system="You are a precise coding assistant. Reply as instructed.",
)

print("\nNormalized assistant text:\n")
print(extract_assistant_text(response))